# Exercise 10: Deep Learning


**This notebook runs in Google Colab or ordinary local Jupyter (Python
3.11 or later) only.** It is not available through this course's in-browser
JupyterLite launcher: PyTorch has no official build for the WebAssembly
environment JupyterLite runs in, and this course's own pinned browser
kernel does not have it available either (verified directly, not assumed
-- see the note at the end of Section 1). Open this notebook directly in
Colab (upload it or open it from Google Drive/GitHub) or in a local
Jupyter installation.

## What this notebook covers

This is Exercise 10 of the Machine Learning for Neuroscience practice
series. The lecture already introduced layers, dense networks, losses,
epochs, and training loops; here you build and train a small one yourself.

In this notebook, you will:

1. load the data and see how many participants and structural features are available
2. split the data, build PyTorch tensors, and inspect a batch
3. define a small fully connected network
4. complete key parts of the training and evaluation loops, then train it
5. evaluate the trained network once on the test set and compare it with a baseline
6. answer questions about what the network is doing and why certain choices matter


**Run the cell below first.** Its code is collapsed (click the `...` to
expand it) because it is one-time setup, not part of the lesson -- but it
still has to run once, before anything else in this notebook, or later
cells will fail with `NameError`. Click it, then press Shift+Enter (or the
Run button), and continue through the notebook in order from there.


In [ ]:
import hashlib
import io
import random
import urllib.request
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

warnings.filterwarnings("ignore", message=r"The (width|height|x|y) parameter as float was deprecated")

_FEATURE_COLUMNS = [
    'fsCT_L_1_ROI', 'fsCT_R_1_ROI', 'fsArea_L_1_ROI', 'fsArea_R_1_ROI',
    'fsVol_L_1_ROI', 'fsVol_R_1_ROI', 'fsLGI_L_1_ROI', 'fsLGI_R_1_ROI',
    'fsCT_L_10d_ROI', 'fsCT_R_10d_ROI', 'fsArea_L_10d_ROI', 'fsArea_R_10d_ROI',
    'fsVol_L_10d_ROI', 'fsVol_R_10d_ROI', 'fsLGI_L_10d_ROI', 'fsLGI_R_10d_ROI',
    'fsCT_L_10pp_ROI', 'fsCT_R_10pp_ROI', 'fsArea_L_10pp_ROI', 'fsArea_R_10pp_ROI',
    'fsVol_L_10pp_ROI', 'fsVol_R_10pp_ROI', 'fsLGI_L_10pp_ROI', 'fsLGI_R_10pp_ROI',
    'fsCT_L_10r_ROI', 'fsCT_R_10r_ROI', 'fsArea_L_10r_ROI', 'fsArea_R_10r_ROI',
    'fsVol_L_10r_ROI', 'fsVol_R_10r_ROI', 'fsLGI_L_10r_ROI', 'fsLGI_R_10r_ROI',
    'fsCT_L_10v_ROI', 'fsCT_R_10v_ROI', 'fsArea_L_10v_ROI', 'fsArea_R_10v_ROI',
    'fsVol_L_10v_ROI', 'fsVol_R_10v_ROI', 'fsLGI_L_10v_ROI', 'fsLGI_R_10v_ROI',
    'fsCT_L_11l_ROI', 'fsCT_R_11l_ROI', 'fsArea_L_11l_ROI', 'fsArea_R_11l_ROI',
    'fsVol_L_11l_ROI', 'fsVol_R_11l_ROI', 'fsLGI_L_11l_ROI', 'fsLGI_R_11l_ROI',
    'fsCT_L_13l_ROI', 'fsCT_R_13l_ROI', 'fsArea_L_13l_ROI', 'fsArea_R_13l_ROI',
    'fsVol_L_13l_ROI', 'fsVol_R_13l_ROI', 'fsLGI_L_13l_ROI', 'fsLGI_R_13l_ROI',
    'fsCT_L_2_ROI', 'fsCT_R_2_ROI', 'fsArea_L_2_ROI', 'fsArea_R_2_ROI',
    'fsVol_L_2_ROI', 'fsVol_R_2_ROI', 'fsLGI_L_2_ROI', 'fsLGI_R_2_ROI',
    'fsCT_L_23c_ROI', 'fsCT_R_23c_ROI', 'fsArea_L_23c_ROI', 'fsArea_R_23c_ROI',
    'fsVol_L_23c_ROI', 'fsVol_R_23c_ROI', 'fsLGI_L_23c_ROI', 'fsLGI_R_23c_ROI',
    'fsCT_L_23d_ROI', 'fsCT_R_23d_ROI', 'fsArea_L_23d_ROI', 'fsArea_R_23d_ROI',
    'fsVol_L_23d_ROI', 'fsVol_R_23d_ROI', 'fsLGI_L_23d_ROI', 'fsLGI_R_23d_ROI',
    'fsCT_L_24dd_ROI', 'fsCT_R_24dd_ROI', 'fsArea_L_24dd_ROI', 'fsArea_R_24dd_ROI',
    'fsVol_L_24dd_ROI', 'fsVol_R_24dd_ROI', 'fsLGI_L_24dd_ROI', 'fsLGI_R_24dd_ROI',
    'fsCT_L_24dv_ROI', 'fsCT_R_24dv_ROI', 'fsArea_L_24dv_ROI', 'fsArea_R_24dv_ROI',
    'fsVol_L_24dv_ROI', 'fsVol_R_24dv_ROI', 'fsLGI_L_24dv_ROI', 'fsLGI_R_24dv_ROI',
    'fsCT_L_25_ROI', 'fsCT_R_25_ROI', 'fsArea_L_25_ROI', 'fsArea_R_25_ROI',
    'fsVol_L_25_ROI', 'fsVol_R_25_ROI', 'fsLGI_L_25_ROI', 'fsLGI_R_25_ROI',
    'fsCT_L_31a_ROI', 'fsCT_R_31a_ROI', 'fsArea_L_31a_ROI', 'fsArea_R_31a_ROI',
    'fsVol_L_31a_ROI', 'fsVol_R_31a_ROI', 'fsLGI_L_31a_ROI', 'fsLGI_R_31a_ROI',
    'fsCT_L_31pd_ROI', 'fsCT_R_31pd_ROI', 'fsArea_L_31pd_ROI', 'fsArea_R_31pd_ROI',
    'fsVol_L_31pd_ROI', 'fsVol_R_31pd_ROI', 'fsLGI_L_31pd_ROI', 'fsLGI_R_31pd_ROI',
    'fsCT_L_31pv_ROI', 'fsCT_R_31pv_ROI', 'fsArea_L_31pv_ROI', 'fsArea_R_31pv_ROI',
    'fsVol_L_31pv_ROI', 'fsVol_R_31pv_ROI', 'fsLGI_L_31pv_ROI', 'fsLGI_R_31pv_ROI',
    'fsCT_L_33pr_ROI', 'fsCT_R_33pr_ROI', 'fsArea_L_33pr_ROI', 'fsArea_R_33pr_ROI',
    'fsVol_L_33pr_ROI', 'fsVol_R_33pr_ROI', 'fsLGI_L_33pr_ROI', 'fsLGI_R_33pr_ROI',
    'fsCT_L_3a_ROI', 'fsCT_R_3a_ROI', 'fsArea_L_3a_ROI', 'fsArea_R_3a_ROI',
    'fsVol_L_3a_ROI', 'fsVol_R_3a_ROI', 'fsLGI_L_3a_ROI', 'fsLGI_R_3a_ROI',
    'fsCT_L_3b_ROI', 'fsCT_R_3b_ROI', 'fsArea_L_3b_ROI', 'fsArea_R_3b_ROI',
    'fsVol_L_3b_ROI', 'fsVol_R_3b_ROI', 'fsLGI_L_3b_ROI', 'fsLGI_R_3b_ROI',
    'fsCT_L_4_ROI', 'fsCT_R_4_ROI', 'fsArea_L_4_ROI', 'fsArea_R_4_ROI',
    'fsVol_L_4_ROI', 'fsVol_R_4_ROI', 'fsLGI_L_4_ROI', 'fsLGI_R_4_ROI',
    'fsCT_L_43_ROI', 'fsCT_R_43_ROI', 'fsArea_L_43_ROI', 'fsArea_R_43_ROI',
    'fsVol_L_43_ROI', 'fsVol_R_43_ROI', 'fsLGI_L_43_ROI', 'fsLGI_R_43_ROI',
    'fsCT_L_44_ROI', 'fsCT_R_44_ROI', 'fsArea_L_44_ROI', 'fsArea_R_44_ROI',
    'fsVol_L_44_ROI', 'fsVol_R_44_ROI', 'fsLGI_L_44_ROI', 'fsLGI_R_44_ROI',
    'fsCT_L_45_ROI', 'fsCT_R_45_ROI', 'fsArea_L_45_ROI', 'fsArea_R_45_ROI',
    'fsVol_L_45_ROI', 'fsVol_R_45_ROI', 'fsLGI_L_45_ROI', 'fsLGI_R_45_ROI',
    'fsCT_L_46_ROI', 'fsCT_R_46_ROI', 'fsArea_L_46_ROI', 'fsArea_R_46_ROI',
    'fsVol_L_46_ROI', 'fsVol_R_46_ROI', 'fsLGI_L_46_ROI', 'fsLGI_R_46_ROI',
    'fsCT_L_47l_ROI', 'fsCT_R_47l_ROI', 'fsArea_L_47l_ROI', 'fsArea_R_47l_ROI',
    'fsVol_L_47l_ROI', 'fsVol_R_47l_ROI', 'fsLGI_L_47l_ROI', 'fsLGI_R_47l_ROI',
    'fsCT_L_47m_ROI', 'fsCT_R_47m_ROI', 'fsArea_L_47m_ROI', 'fsArea_R_47m_ROI',
    'fsVol_L_47m_ROI', 'fsVol_R_47m_ROI', 'fsLGI_L_47m_ROI', 'fsLGI_R_47m_ROI',
    'fsCT_L_47s_ROI', 'fsCT_R_47s_ROI', 'fsArea_L_47s_ROI', 'fsArea_R_47s_ROI',
    'fsVol_L_47s_ROI', 'fsVol_R_47s_ROI', 'fsLGI_L_47s_ROI', 'fsLGI_R_47s_ROI',
    'fsCT_L_5L_ROI', 'fsCT_R_5R_ROI', 'fsArea_L_5L_ROI', 'fsArea_R_5R_ROI',
    'fsVol_L_5L_ROI', 'fsVol_R_5R_ROI', 'fsLGI_L_5L_ROI', 'fsLGI_R_5R_ROI',
    'fsCT_L_52_ROI', 'fsCT_R_52_ROI', 'fsArea_L_52_ROI', 'fsArea_R_52_ROI',
    'fsVol_L_52_ROI', 'fsVol_R_52_ROI', 'fsLGI_L_52_ROI', 'fsLGI_R_52_ROI',
    'fsCT_L_55b_ROI', 'fsCT_R_55b_ROI', 'fsArea_L_55b_ROI', 'fsArea_R_55b_ROI',
    'fsVol_L_55b_ROI', 'fsVol_R_55b_ROI', 'fsLGI_L_55b_ROI', 'fsLGI_R_55b_ROI',
    'fsCT_L_5m_ROI', 'fsCT_R_5m_ROI', 'fsArea_L_5m_ROI', 'fsArea_R_5m_ROI',
    'fsVol_L_5m_ROI', 'fsVol_R_5m_ROI', 'fsLGI_L_5m_ROI', 'fsLGI_R_5m_ROI',
    'fsCT_L_5mv_ROI', 'fsCT_R_5mv_ROI', 'fsArea_L_5mv_ROI', 'fsArea_R_5mv_ROI',
    'fsVol_L_5mv_ROI', 'fsVol_R_5mv_ROI', 'fsLGI_L_5mv_ROI', 'fsLGI_R_5mv_ROI',
    'fsCT_L_6a_ROI', 'fsCT_R_6a_ROI', 'fsArea_L_6a_ROI', 'fsArea_R_6a_ROI',
    'fsVol_L_6a_ROI', 'fsVol_R_6a_ROI', 'fsLGI_L_6a_ROI', 'fsLGI_R_6a_ROI',
    'fsCT_L_6d_ROI', 'fsCT_R_6d_ROI', 'fsArea_L_6d_ROI', 'fsArea_R_6d_ROI',
    'fsVol_L_6d_ROI', 'fsVol_R_6d_ROI', 'fsLGI_L_6d_ROI', 'fsLGI_R_6d_ROI',
    'fsCT_L_6ma_ROI', 'fsCT_R_6ma_ROI', 'fsArea_L_6ma_ROI', 'fsArea_R_6ma_ROI',
    'fsVol_L_6ma_ROI', 'fsVol_R_6ma_ROI', 'fsLGI_L_6ma_ROI', 'fsLGI_R_6ma_ROI',
    'fsCT_L_6mp_ROI', 'fsCT_R_6mp_ROI', 'fsArea_L_6mp_ROI', 'fsArea_R_6mp_ROI',
    'fsVol_L_6mp_ROI', 'fsVol_R_6mp_ROI', 'fsLGI_L_6mp_ROI', 'fsLGI_R_6mp_ROI',
    'fsCT_L_6r_ROI', 'fsCT_R_6r_ROI', 'fsArea_L_6r_ROI', 'fsArea_R_6r_ROI',
    'fsVol_L_6r_ROI', 'fsVol_R_6r_ROI', 'fsLGI_L_6r_ROI', 'fsLGI_R_6r_ROI',
    'fsCT_L_6v_ROI', 'fsCT_R_6v_ROI', 'fsArea_L_6v_ROI', 'fsArea_R_6v_ROI',
    'fsVol_L_6v_ROI', 'fsVol_R_6v_ROI', 'fsLGI_L_6v_ROI', 'fsLGI_R_6v_ROI',
    'fsCT_L_7AL_ROI', 'fsCT_R_7AL_ROI', 'fsArea_L_7AL_ROI', 'fsArea_R_7AL_ROI',
    'fsVol_L_7AL_ROI', 'fsVol_R_7AL_ROI', 'fsLGI_L_7AL_ROI', 'fsLGI_R_7AL_ROI',
    'fsCT_L_7Am_ROI', 'fsCT_R_7Am_ROI', 'fsArea_L_7Am_ROI', 'fsArea_R_7Am_ROI',
    'fsVol_L_7Am_ROI', 'fsVol_R_7Am_ROI', 'fsLGI_L_7Am_ROI', 'fsLGI_R_7Am_ROI',
    'fsCT_L_7PC_ROI', 'fsCT_R_7PC_ROI', 'fsArea_L_7PC_ROI', 'fsArea_R_7PC_ROI',
    'fsVol_L_7PC_ROI', 'fsVol_R_7PC_ROI', 'fsLGI_L_7PC_ROI', 'fsLGI_R_7PC_ROI',
    'fsCT_L_7PL_ROI', 'fsCT_R_7PL_ROI', 'fsArea_L_7PL_ROI', 'fsArea_R_7PL_ROI',
    'fsVol_L_7PL_ROI', 'fsVol_R_7PL_ROI', 'fsLGI_L_7PL_ROI', 'fsLGI_R_7PL_ROI',
    'fsCT_L_7Pm_ROI', 'fsCT_R_7Pm_ROI', 'fsArea_L_7Pm_ROI', 'fsArea_R_7Pm_ROI',
    'fsVol_L_7Pm_ROI', 'fsVol_R_7Pm_ROI', 'fsLGI_L_7Pm_ROI', 'fsLGI_R_7Pm_ROI',
    'fsCT_L_7m_ROI', 'fsCT_R_7m_ROI', 'fsArea_L_7m_ROI', 'fsArea_R_7m_ROI',
    'fsVol_L_7m_ROI', 'fsVol_R_7m_ROI', 'fsLGI_L_7m_ROI', 'fsLGI_R_7m_ROI',
    'fsCT_L_8Ad_ROI', 'fsCT_R_8Ad_ROI', 'fsArea_L_8Ad_ROI', 'fsArea_R_8Ad_ROI',
    'fsVol_L_8Ad_ROI', 'fsVol_R_8Ad_ROI', 'fsLGI_L_8Ad_ROI', 'fsLGI_R_8Ad_ROI',
    'fsCT_L_8Av_ROI', 'fsCT_R_8Av_ROI', 'fsArea_L_8Av_ROI', 'fsArea_R_8Av_ROI',
    'fsVol_L_8Av_ROI', 'fsVol_R_8Av_ROI', 'fsLGI_L_8Av_ROI', 'fsLGI_R_8Av_ROI',
    'fsCT_L_8BL_ROI', 'fsCT_R_8BL_ROI', 'fsArea_L_8BL_ROI', 'fsArea_R_8BL_ROI',
    'fsVol_L_8BL_ROI', 'fsVol_R_8BL_ROI', 'fsLGI_L_8BL_ROI', 'fsLGI_R_8BL_ROI',
    'fsCT_L_8BM_ROI', 'fsCT_R_8BM_ROI', 'fsArea_L_8BM_ROI', 'fsArea_R_8BM_ROI',
    'fsVol_L_8BM_ROI', 'fsVol_R_8BM_ROI', 'fsLGI_L_8BM_ROI', 'fsLGI_R_8BM_ROI',
    'fsCT_L_8C_ROI', 'fsCT_R_8C_ROI', 'fsArea_L_8C_ROI', 'fsArea_R_8C_ROI',
    'fsVol_L_8C_ROI', 'fsVol_R_8C_ROI', 'fsLGI_L_8C_ROI', 'fsLGI_R_8C_ROI',
    'fsCT_L_9-46d_ROI', 'fsCT_R_9-46d_ROI', 'fsArea_L_9-46d_ROI', 'fsArea_R_9-46d_ROI',
    'fsVol_L_9-46d_ROI', 'fsVol_R_9-46d_ROI', 'fsLGI_L_9-46d_ROI', 'fsLGI_R_9-46d_ROI',
    'fsCT_L_9a_ROI', 'fsCT_R_9a_ROI', 'fsArea_L_9a_ROI', 'fsArea_R_9a_ROI',
    'fsVol_L_9a_ROI', 'fsVol_R_9a_ROI', 'fsLGI_L_9a_ROI', 'fsLGI_R_9a_ROI',
    'fsCT_L_9m_ROI', 'fsCT_R_9m_ROI', 'fsArea_L_9m_ROI', 'fsArea_R_9m_ROI',
    'fsVol_L_9m_ROI', 'fsVol_R_9m_ROI', 'fsLGI_L_9m_ROI', 'fsLGI_R_9m_ROI',
    'fsCT_L_9p_ROI', 'fsCT_R_9p_ROI', 'fsArea_L_9p_ROI', 'fsArea_R_9p_ROI',
    'fsVol_L_9p_ROI', 'fsVol_R_9p_ROI', 'fsLGI_L_9p_ROI', 'fsLGI_R_9p_ROI',
    'fsCT_L_A1_ROI', 'fsCT_R_A1_ROI', 'fsArea_L_A1_ROI', 'fsArea_R_A1_ROI',
    'fsVol_L_A1_ROI', 'fsVol_R_A1_ROI', 'fsLGI_L_A1_ROI', 'fsLGI_R_A1_ROI',
    'fsCT_L_A4_ROI', 'fsCT_R_A4_ROI', 'fsArea_L_A4_ROI', 'fsArea_R_A4_ROI',
    'fsVol_L_A4_ROI', 'fsVol_R_A4_ROI', 'fsLGI_L_A4_ROI', 'fsLGI_R_A4_ROI',
    'fsCT_L_A5_ROI', 'fsCT_R_A5_ROI', 'fsArea_L_A5_ROI', 'fsArea_R_A5_ROI',
    'fsVol_L_A5_ROI', 'fsVol_R_A5_ROI', 'fsLGI_L_A5_ROI', 'fsLGI_R_A5_ROI',
    'fsCT_L_AAIC_ROI', 'fsCT_R_AAIC_ROI', 'fsArea_L_AAIC_ROI', 'fsArea_R_AAIC_ROI',
    'fsVol_L_AAIC_ROI', 'fsVol_R_AAIC_ROI', 'fsLGI_L_AAIC_ROI', 'fsLGI_R_AAIC_ROI',
    'fsCT_L_AIP_ROI', 'fsCT_R_AIP_ROI', 'fsArea_L_AIP_ROI', 'fsArea_R_AIP_ROI',
    'fsVol_L_AIP_ROI', 'fsVol_R_AIP_ROI', 'fsLGI_L_AIP_ROI', 'fsLGI_R_AIP_ROI',
    'fsCT_L_AVI_ROI', 'fsCT_R_AVI_ROI', 'fsArea_L_AVI_ROI', 'fsArea_R_AVI_ROI',
    'fsVol_L_AVI_ROI', 'fsVol_R_AVI_ROI', 'fsLGI_L_AVI_ROI', 'fsLGI_R_AVI_ROI',
    'fsCT_L_DVT_ROI', 'fsCT_R_DVT_ROI', 'fsArea_L_DVT_ROI', 'fsArea_R_DVT_ROI',
    'fsVol_L_DVT_ROI', 'fsVol_R_DVT_ROI', 'fsLGI_L_DVT_ROI', 'fsLGI_R_DVT_ROI',
    'fsCT_L_EC_ROI', 'fsCT_R_EC_ROI', 'fsArea_L_EC_ROI', 'fsArea_R_EC_ROI',
    'fsVol_L_EC_ROI', 'fsVol_R_EC_ROI', 'fsLGI_L_EC_ROI', 'fsLGI_R_EC_ROI',
    'fsCT_L_FEF_ROI', 'fsCT_R_FEF_ROI', 'fsArea_L_FEF_ROI', 'fsArea_R_FEF_ROI',
    'fsVol_L_FEF_ROI', 'fsVol_R_FEF_ROI', 'fsLGI_L_FEF_ROI', 'fsLGI_R_FEF_ROI',
    'fsCT_L_FFC_ROI', 'fsCT_R_FFC_ROI', 'fsArea_L_FFC_ROI', 'fsArea_R_FFC_ROI',
    'fsVol_L_FFC_ROI', 'fsVol_R_FFC_ROI', 'fsLGI_L_FFC_ROI', 'fsLGI_R_FFC_ROI',
    'fsCT_L_FOP1_ROI', 'fsCT_R_FOP1_ROI', 'fsArea_L_FOP1_ROI', 'fsArea_R_FOP1_ROI',
    'fsVol_L_FOP1_ROI', 'fsVol_R_FOP1_ROI', 'fsLGI_L_FOP1_ROI', 'fsLGI_R_FOP1_ROI',
    'fsCT_L_FOP2_ROI', 'fsCT_R_FOP2_ROI', 'fsArea_L_FOP2_ROI', 'fsArea_R_FOP2_ROI',
    'fsVol_L_FOP2_ROI', 'fsVol_R_FOP2_ROI', 'fsLGI_L_FOP2_ROI', 'fsLGI_R_FOP2_ROI',
    'fsCT_L_FOP3_ROI', 'fsCT_R_FOP3_ROI', 'fsArea_L_FOP3_ROI', 'fsArea_R_FOP3_ROI',
    'fsVol_L_FOP3_ROI', 'fsVol_R_FOP3_ROI', 'fsLGI_L_FOP3_ROI', 'fsLGI_R_FOP3_ROI',
    'fsCT_L_FOP4_ROI', 'fsCT_R_FOP4_ROI', 'fsArea_L_FOP4_ROI', 'fsArea_R_FOP4_ROI',
    'fsVol_L_FOP4_ROI', 'fsVol_R_FOP4_ROI', 'fsLGI_L_FOP4_ROI', 'fsLGI_R_FOP4_ROI',
    'fsCT_L_FOP5_ROI', 'fsCT_R_FOP5_ROI', 'fsArea_L_FOP5_ROI', 'fsArea_R_FOP5_ROI',
    'fsVol_L_FOP5_ROI', 'fsVol_R_FOP5_ROI', 'fsLGI_L_FOP5_ROI', 'fsLGI_R_FOP5_ROI',
    'fsCT_L_FST_ROI', 'fsCT_R_FST_ROI', 'fsArea_L_FST_ROI', 'fsArea_R_FST_ROI',
    'fsVol_L_FST_ROI', 'fsVol_R_FST_ROI', 'fsLGI_L_FST_ROI', 'fsLGI_R_FST_ROI',
    'fsCT_L_H_ROI', 'fsCT_R_H_ROI', 'fsArea_L_H_ROI', 'fsArea_R_H_ROI',
    'fsVol_L_H_ROI', 'fsVol_R_H_ROI', 'fsLGI_L_H_ROI', 'fsLGI_R_H_ROI',
    'fsCT_L_IFJa_ROI', 'fsCT_R_IFJa_ROI', 'fsArea_L_IFJa_ROI', 'fsArea_R_IFJa_ROI',
    'fsVol_L_IFJa_ROI', 'fsVol_R_IFJa_ROI', 'fsLGI_L_IFJa_ROI', 'fsLGI_R_IFJa_ROI',
    'fsCT_L_IFJp_ROI', 'fsCT_R_IFJp_ROI', 'fsArea_L_IFJp_ROI', 'fsArea_R_IFJp_ROI',
    'fsVol_L_IFJp_ROI', 'fsVol_R_IFJp_ROI', 'fsLGI_L_IFJp_ROI', 'fsLGI_R_IFJp_ROI',
    'fsCT_L_IFSa_ROI', 'fsCT_R_IFSa_ROI', 'fsArea_L_IFSa_ROI', 'fsArea_R_IFSa_ROI',
    'fsVol_L_IFSa_ROI', 'fsVol_R_IFSa_ROI', 'fsLGI_L_IFSa_ROI', 'fsLGI_R_IFSa_ROI',
    'fsCT_L_IFSp_ROI', 'fsCT_R_IFSp_ROI', 'fsArea_L_IFSp_ROI', 'fsArea_R_IFSp_ROI',
    'fsVol_L_IFSp_ROI', 'fsVol_R_IFSp_ROI', 'fsLGI_L_IFSp_ROI', 'fsLGI_R_IFSp_ROI',
    'fsCT_L_IP0_ROI', 'fsCT_R_IP0_ROI', 'fsArea_L_IP0_ROI', 'fsArea_R_IP0_ROI',
    'fsVol_L_IP0_ROI', 'fsVol_R_IP0_ROI', 'fsLGI_L_IP0_ROI', 'fsLGI_R_IP0_ROI',
    'fsCT_L_IP1_ROI', 'fsCT_R_IP1_ROI', 'fsArea_L_IP1_ROI', 'fsArea_R_IP1_ROI',
    'fsVol_L_IP1_ROI', 'fsVol_R_IP1_ROI', 'fsLGI_L_IP1_ROI', 'fsLGI_R_IP1_ROI',
    'fsCT_L_IP2_ROI', 'fsCT_R_IP2_ROI', 'fsArea_L_IP2_ROI', 'fsArea_R_IP2_ROI',
    'fsVol_L_IP2_ROI', 'fsVol_R_IP2_ROI', 'fsLGI_L_IP2_ROI', 'fsLGI_R_IP2_ROI',
    'fsCT_L_IPS1_ROI', 'fsCT_R_IPS1_ROI', 'fsArea_L_IPS1_ROI', 'fsArea_R_IPS1_ROI',
    'fsVol_L_IPS1_ROI', 'fsVol_R_IPS1_ROI', 'fsLGI_L_IPS1_ROI', 'fsLGI_R_IPS1_ROI',
    'fsCT_L_Ig_ROI', 'fsCT_R_Ig_ROI', 'fsArea_L_Ig_ROI', 'fsArea_R_Ig_ROI',
    'fsVol_L_Ig_ROI', 'fsVol_R_Ig_ROI', 'fsLGI_L_Ig_ROI', 'fsLGI_R_Ig_ROI',
    'fsCT_L_LBelt_ROI', 'fsCT_R_LBelt_ROI', 'fsArea_L_LBelt_ROI', 'fsArea_R_LBelt_ROI',
    'fsVol_L_LBelt_ROI', 'fsVol_R_LBelt_ROI', 'fsLGI_L_LBelt_ROI', 'fsLGI_R_LBelt_ROI',
    'fsCT_L_LIPd_ROI', 'fsCT_R_LIPd_ROI', 'fsArea_L_LIPd_ROI', 'fsArea_R_LIPd_ROI',
    'fsVol_L_LIPd_ROI', 'fsVol_R_LIPd_ROI', 'fsLGI_L_LIPd_ROI', 'fsLGI_R_LIPd_ROI',
    'fsCT_L_LIPv_ROI', 'fsCT_R_LIPv_ROI', 'fsArea_L_LIPv_ROI', 'fsArea_R_LIPv_ROI',
    'fsVol_L_LIPv_ROI', 'fsVol_R_LIPv_ROI', 'fsLGI_L_LIPv_ROI', 'fsLGI_R_LIPv_ROI',
    'fsCT_L_LO1_ROI', 'fsCT_R_LO1_ROI', 'fsArea_L_LO1_ROI', 'fsArea_R_LO1_ROI',
    'fsVol_L_LO1_ROI', 'fsVol_R_LO1_ROI', 'fsLGI_L_LO1_ROI', 'fsLGI_R_LO1_ROI',
    'fsCT_L_LO2_ROI', 'fsCT_R_LO2_ROI', 'fsArea_L_LO2_ROI', 'fsArea_R_LO2_ROI',
    'fsVol_L_LO2_ROI', 'fsVol_R_LO2_ROI', 'fsLGI_L_LO2_ROI', 'fsLGI_R_LO2_ROI',
    'fsCT_L_LO3_ROI', 'fsCT_R_LO3_ROI', 'fsArea_L_LO3_ROI', 'fsArea_R_LO3_ROI',
    'fsVol_L_LO3_ROI', 'fsVol_R_LO3_ROI', 'fsLGI_L_LO3_ROI', 'fsLGI_R_LO3_ROI',
    'fsCT_L_MBelt_ROI', 'fsCT_R_MBelt_ROI', 'fsArea_L_MBelt_ROI', 'fsArea_R_MBelt_ROI',
    'fsVol_L_MBelt_ROI', 'fsVol_R_MBelt_ROI', 'fsLGI_L_MBelt_ROI', 'fsLGI_R_MBelt_ROI',
    'fsCT_L_MI_ROI', 'fsCT_R_MI_ROI', 'fsArea_L_MI_ROI', 'fsArea_R_MI_ROI',
    'fsVol_L_MI_ROI', 'fsVol_R_MI_ROI', 'fsLGI_L_MI_ROI', 'fsLGI_R_MI_ROI',
    'fsCT_L_MIP_ROI', 'fsCT_R_MIP_ROI', 'fsArea_L_MIP_ROI', 'fsArea_R_MIP_ROI',
    'fsVol_L_MIP_ROI', 'fsVol_R_MIP_ROI', 'fsLGI_L_MIP_ROI', 'fsLGI_R_MIP_ROI',
    'fsCT_L_MST_ROI', 'fsCT_R_MST_ROI', 'fsArea_L_MST_ROI', 'fsArea_R_MST_ROI',
    'fsVol_L_MST_ROI', 'fsVol_R_MST_ROI', 'fsLGI_L_MST_ROI', 'fsLGI_R_MST_ROI',
    'fsCT_L_MT_ROI', 'fsCT_R_MT_ROI', 'fsArea_L_MT_ROI', 'fsArea_R_MT_ROI',
    'fsVol_L_MT_ROI', 'fsVol_R_MT_ROI', 'fsLGI_L_MT_ROI', 'fsLGI_R_MT_ROI',
    'fsCT_L_OFC_ROI', 'fsCT_R_OFC_ROI', 'fsArea_L_OFC_ROI', 'fsArea_R_OFC_ROI',
    'fsVol_L_OFC_ROI', 'fsVol_R_OFC_ROI', 'fsLGI_L_OFC_ROI', 'fsLGI_R_OFC_ROI',
    'fsCT_L_OP1_ROI', 'fsCT_R_OP1_ROI', 'fsArea_L_OP1_ROI', 'fsArea_R_OP1_ROI',
    'fsVol_L_OP1_ROI', 'fsVol_R_OP1_ROI', 'fsLGI_L_OP1_ROI', 'fsLGI_R_OP1_ROI',
    'fsCT_L_OP2-3_ROI', 'fsCT_R_OP2-3_ROI', 'fsArea_L_OP2-3_ROI', 'fsArea_R_OP2-3_ROI',
    'fsVol_L_OP2-3_ROI', 'fsVol_R_OP2-3_ROI', 'fsLGI_L_OP2-3_ROI', 'fsLGI_R_OP2-3_ROI',
    'fsCT_L_OP4_ROI', 'fsCT_R_OP4_ROI', 'fsArea_L_OP4_ROI', 'fsArea_R_OP4_ROI',
    'fsVol_L_OP4_ROI', 'fsVol_R_OP4_ROI', 'fsLGI_L_OP4_ROI', 'fsLGI_R_OP4_ROI',
    'fsCT_L_PBelt_ROI', 'fsCT_R_PBelt_ROI', 'fsArea_L_PBelt_ROI', 'fsArea_R_PBelt_ROI',
    'fsVol_L_PBelt_ROI', 'fsVol_R_PBelt_ROI', 'fsLGI_L_PBelt_ROI', 'fsLGI_R_PBelt_ROI',
    'fsCT_L_PCV_ROI', 'fsCT_R_PCV_ROI', 'fsArea_L_PCV_ROI', 'fsArea_R_PCV_ROI',
    'fsVol_L_PCV_ROI', 'fsVol_R_PCV_ROI', 'fsLGI_L_PCV_ROI', 'fsLGI_R_PCV_ROI',
    'fsCT_L_PEF_ROI', 'fsCT_R_PEF_ROI', 'fsArea_L_PEF_ROI', 'fsArea_R_PEF_ROI',
    'fsVol_L_PEF_ROI', 'fsVol_R_PEF_ROI', 'fsLGI_L_PEF_ROI', 'fsLGI_R_PEF_ROI',
    'fsCT_L_PF_ROI', 'fsCT_R_PF_ROI', 'fsArea_L_PF_ROI', 'fsArea_R_PF_ROI',
    'fsVol_L_PF_ROI', 'fsVol_R_PF_ROI', 'fsLGI_L_PF_ROI', 'fsLGI_R_PF_ROI',
    'fsCT_L_PFcm_ROI', 'fsCT_R_PFcm_ROI', 'fsArea_L_PFcm_ROI', 'fsArea_R_PFcm_ROI',
    'fsVol_L_PFcm_ROI', 'fsVol_R_PFcm_ROI', 'fsLGI_L_PFcm_ROI', 'fsLGI_R_PFcm_ROI',
    'fsCT_L_PFm_ROI', 'fsCT_R_PFm_ROI', 'fsArea_L_PFm_ROI', 'fsArea_R_PFm_ROI',
    'fsVol_L_PFm_ROI', 'fsVol_R_PFm_ROI', 'fsLGI_L_PFm_ROI', 'fsLGI_R_PFm_ROI',
    'fsCT_L_PFop_ROI', 'fsCT_R_PFop_ROI', 'fsArea_L_PFop_ROI', 'fsArea_R_PFop_ROI',
    'fsVol_L_PFop_ROI', 'fsVol_R_PFop_ROI', 'fsLGI_L_PFop_ROI', 'fsLGI_R_PFop_ROI',
    'fsCT_L_PFt_ROI', 'fsCT_R_PFt_ROI', 'fsArea_L_PFt_ROI', 'fsArea_R_PFt_ROI',
    'fsVol_L_PFt_ROI', 'fsVol_R_PFt_ROI', 'fsLGI_L_PFt_ROI', 'fsLGI_R_PFt_ROI',
    'fsCT_L_PGi_ROI', 'fsCT_R_PGi_ROI', 'fsArea_L_PGi_ROI', 'fsArea_R_PGi_ROI',
    'fsVol_L_PGi_ROI', 'fsVol_R_PGi_ROI', 'fsLGI_L_PGi_ROI', 'fsLGI_R_PGi_ROI',
    'fsCT_L_PGp_ROI', 'fsCT_R_PGp_ROI', 'fsArea_L_PGp_ROI', 'fsArea_R_PGp_ROI',
    'fsVol_L_PGp_ROI', 'fsVol_R_PGp_ROI', 'fsLGI_L_PGp_ROI', 'fsLGI_R_PGp_ROI',
    'fsCT_L_PGs_ROI', 'fsCT_R_PGs_ROI', 'fsArea_L_PGs_ROI', 'fsArea_R_PGs_ROI',
    'fsVol_L_PGs_ROI', 'fsVol_R_PGs_ROI', 'fsLGI_L_PGs_ROI', 'fsLGI_R_PGs_ROI',
    'fsCT_L_PH_ROI', 'fsCT_R_PH_ROI', 'fsArea_L_PH_ROI', 'fsArea_R_PH_ROI',
    'fsVol_L_PH_ROI', 'fsVol_R_PH_ROI', 'fsLGI_L_PH_ROI', 'fsLGI_R_PH_ROI',
    'fsCT_L_PHA1_ROI', 'fsCT_R_PHA1_ROI', 'fsArea_L_PHA1_ROI', 'fsArea_R_PHA1_ROI',
    'fsVol_L_PHA1_ROI', 'fsVol_R_PHA1_ROI', 'fsLGI_L_PHA1_ROI', 'fsLGI_R_PHA1_ROI',
    'fsCT_L_PHA2_ROI', 'fsCT_R_PHA2_ROI', 'fsArea_L_PHA2_ROI', 'fsArea_R_PHA2_ROI',
    'fsVol_L_PHA2_ROI', 'fsVol_R_PHA2_ROI', 'fsLGI_L_PHA2_ROI', 'fsLGI_R_PHA2_ROI',
    'fsCT_L_PHA3_ROI', 'fsCT_R_PHA3_ROI', 'fsArea_L_PHA3_ROI', 'fsArea_R_PHA3_ROI',
    'fsVol_L_PHA3_ROI', 'fsVol_R_PHA3_ROI', 'fsLGI_L_PHA3_ROI', 'fsLGI_R_PHA3_ROI',
    'fsCT_L_PHT_ROI', 'fsCT_R_PHT_ROI', 'fsArea_L_PHT_ROI', 'fsArea_R_PHT_ROI',
    'fsVol_L_PHT_ROI', 'fsVol_R_PHT_ROI', 'fsLGI_L_PHT_ROI', 'fsLGI_R_PHT_ROI',
    'fsCT_L_PI_ROI', 'fsCT_R_PI_ROI', 'fsArea_L_PI_ROI', 'fsArea_R_PI_ROI',
    'fsVol_L_PI_ROI', 'fsVol_R_PI_ROI', 'fsLGI_L_PI_ROI', 'fsLGI_R_PI_ROI',
    'fsCT_L_PIT_ROI', 'fsCT_R_PIT_ROI', 'fsArea_L_PIT_ROI', 'fsArea_R_PIT_ROI',
    'fsVol_L_PIT_ROI', 'fsVol_R_PIT_ROI', 'fsLGI_L_PIT_ROI', 'fsLGI_R_PIT_ROI',
    'fsCT_L_POS1_ROI', 'fsCT_R_POS1_ROI', 'fsArea_L_POS1_ROI', 'fsArea_R_POS1_ROI',
    'fsVol_L_POS1_ROI', 'fsVol_R_POS1_ROI', 'fsLGI_L_POS1_ROI', 'fsLGI_R_POS1_ROI',
    'fsCT_L_POS2_ROI', 'fsCT_R_POS2_ROI', 'fsArea_L_POS2_ROI', 'fsArea_R_POS2_ROI',
    'fsVol_L_POS2_ROI', 'fsVol_R_POS2_ROI', 'fsLGI_L_POS2_ROI', 'fsLGI_R_POS2_ROI',
    'fsCT_L_PSL_ROI', 'fsCT_R_PSL_ROI', 'fsArea_L_PSL_ROI', 'fsArea_R_PSL_ROI',
    'fsVol_L_PSL_ROI', 'fsVol_R_PSL_ROI', 'fsLGI_L_PSL_ROI', 'fsLGI_R_PSL_ROI',
    'fsCT_L_PeEc_ROI', 'fsCT_R_PeEc_ROI', 'fsArea_L_PeEc_ROI', 'fsArea_R_PeEc_ROI',
    'fsVol_L_PeEc_ROI', 'fsVol_R_PeEc_ROI', 'fsLGI_L_PeEc_ROI', 'fsLGI_R_PeEc_ROI',
    'fsCT_L_Pir_ROI', 'fsCT_R_Pir_ROI', 'fsArea_L_Pir_ROI', 'fsArea_R_Pir_ROI',
    'fsVol_L_Pir_ROI', 'fsVol_R_Pir_ROI', 'fsLGI_L_Pir_ROI', 'fsLGI_R_Pir_ROI',
    'fsCT_L_PoI1_ROI', 'fsCT_R_PoI1_ROI', 'fsArea_L_PoI1_ROI', 'fsArea_R_PoI1_ROI',
    'fsVol_L_PoI1_ROI', 'fsVol_R_PoI1_ROI', 'fsLGI_L_PoI1_ROI', 'fsLGI_R_PoI1_ROI',
    'fsCT_L_PoI2_ROI', 'fsCT_R_PoI2_ROI', 'fsArea_L_PoI2_ROI', 'fsArea_R_PoI2_ROI',
    'fsVol_L_PoI2_ROI', 'fsVol_R_PoI2_ROI', 'fsLGI_L_PoI2_ROI', 'fsLGI_R_PoI2_ROI',
    'fsCT_L_PreS_ROI', 'fsCT_R_PreS_ROI', 'fsArea_L_PreS_ROI', 'fsArea_R_PreS_ROI',
    'fsVol_L_PreS_ROI', 'fsVol_R_PreS_ROI', 'fsLGI_L_PreS_ROI', 'fsLGI_R_PreS_ROI',
    'fsCT_L_ProS_ROI', 'fsCT_R_ProS_ROI', 'fsArea_L_ProS_ROI', 'fsArea_R_ProS_ROI',
    'fsVol_L_ProS_ROI', 'fsVol_R_ProS_ROI', 'fsLGI_L_ProS_ROI', 'fsLGI_R_ProS_ROI',
    'fsCT_L_RI_ROI', 'fsCT_R_RI_ROI', 'fsArea_L_RI_ROI', 'fsArea_R_RI_ROI',
    'fsVol_L_RI_ROI', 'fsVol_R_RI_ROI', 'fsLGI_L_RI_ROI', 'fsLGI_R_RI_ROI',
    'fsCT_L_RSC_ROI', 'fsCT_R_RSC_ROI', 'fsArea_L_RSC_ROI', 'fsArea_R_RSC_ROI',
    'fsVol_L_RSC_ROI', 'fsVol_R_RSC_ROI', 'fsLGI_L_RSC_ROI', 'fsLGI_R_RSC_ROI',
    'fsCT_L_SCEF_ROI', 'fsCT_R_SCEF_ROI', 'fsArea_L_SCEF_ROI', 'fsArea_R_SCEF_ROI',
    'fsVol_L_SCEF_ROI', 'fsVol_R_SCEF_ROI', 'fsLGI_L_SCEF_ROI', 'fsLGI_R_SCEF_ROI',
    'fsCT_L_SFL_ROI', 'fsCT_R_SFL_ROI', 'fsArea_L_SFL_ROI', 'fsArea_R_SFL_ROI',
    'fsVol_L_SFL_ROI', 'fsVol_R_SFL_ROI', 'fsLGI_L_SFL_ROI', 'fsLGI_R_SFL_ROI',
    'fsCT_L_STGa_ROI', 'fsCT_R_STGa_ROI', 'fsArea_L_STGa_ROI', 'fsArea_R_STGa_ROI',
    'fsVol_L_STGa_ROI', 'fsVol_R_STGa_ROI', 'fsLGI_L_STGa_ROI', 'fsLGI_R_STGa_ROI',
    'fsCT_L_STSda_ROI', 'fsCT_R_STSda_ROI', 'fsArea_L_STSda_ROI', 'fsArea_R_STSda_ROI',
    'fsVol_L_STSda_ROI', 'fsVol_R_STSda_ROI', 'fsLGI_L_STSda_ROI', 'fsLGI_R_STSda_ROI',
    'fsCT_L_STSdp_ROI', 'fsCT_R_STSdp_ROI', 'fsArea_L_STSdp_ROI', 'fsArea_R_STSdp_ROI',
    'fsVol_L_STSdp_ROI', 'fsVol_R_STSdp_ROI', 'fsLGI_L_STSdp_ROI', 'fsLGI_R_STSdp_ROI',
    'fsCT_L_STSva_ROI', 'fsCT_R_STSva_ROI', 'fsArea_L_STSva_ROI', 'fsArea_R_STSva_ROI',
    'fsVol_L_STSva_ROI', 'fsVol_R_STSva_ROI', 'fsLGI_L_STSva_ROI', 'fsLGI_R_STSva_ROI',
    'fsCT_L_STSvp_ROI', 'fsCT_R_STSvp_ROI', 'fsArea_L_STSvp_ROI', 'fsArea_R_STSvp_ROI',
    'fsVol_L_STSvp_ROI', 'fsVol_R_STSvp_ROI', 'fsLGI_L_STSvp_ROI', 'fsLGI_R_STSvp_ROI',
    'fsCT_L_STV_ROI', 'fsCT_R_STV_ROI', 'fsArea_L_STV_ROI', 'fsArea_R_STV_ROI',
    'fsVol_L_STV_ROI', 'fsVol_R_STV_ROI', 'fsLGI_L_STV_ROI', 'fsLGI_R_STV_ROI',
    'fsCT_L_TA2_ROI', 'fsCT_R_TA2_ROI', 'fsArea_L_TA2_ROI', 'fsArea_R_TA2_ROI',
    'fsVol_L_TA2_ROI', 'fsVol_R_TA2_ROI', 'fsLGI_L_TA2_ROI', 'fsLGI_R_TA2_ROI',
    'fsCT_L_TE1a_ROI', 'fsCT_R_TE1a_ROI', 'fsArea_L_TE1a_ROI', 'fsArea_R_TE1a_ROI',
    'fsVol_L_TE1a_ROI', 'fsVol_R_TE1a_ROI', 'fsLGI_L_TE1a_ROI', 'fsLGI_R_TE1a_ROI',
    'fsCT_L_TE1m_ROI', 'fsCT_R_TE1m_ROI', 'fsArea_L_TE1m_ROI', 'fsArea_R_TE1m_ROI',
    'fsVol_L_TE1m_ROI', 'fsVol_R_TE1m_ROI', 'fsLGI_L_TE1m_ROI', 'fsLGI_R_TE1m_ROI',
    'fsCT_L_TE1p_ROI', 'fsCT_R_TE1p_ROI', 'fsArea_L_TE1p_ROI', 'fsArea_R_TE1p_ROI',
    'fsVol_L_TE1p_ROI', 'fsVol_R_TE1p_ROI', 'fsLGI_L_TE1p_ROI', 'fsLGI_R_TE1p_ROI',
    'fsCT_L_TE2a_ROI', 'fsCT_R_TE2a_ROI', 'fsArea_L_TE2a_ROI', 'fsArea_R_TE2a_ROI',
    'fsVol_L_TE2a_ROI', 'fsVol_R_TE2a_ROI', 'fsLGI_L_TE2a_ROI', 'fsLGI_R_TE2a_ROI',
    'fsCT_L_TE2p_ROI', 'fsCT_R_TE2p_ROI', 'fsArea_L_TE2p_ROI', 'fsArea_R_TE2p_ROI',
    'fsVol_L_TE2p_ROI', 'fsVol_R_TE2p_ROI', 'fsLGI_L_TE2p_ROI', 'fsLGI_R_TE2p_ROI',
    'fsCT_L_TF_ROI', 'fsCT_R_TF_ROI', 'fsArea_L_TF_ROI', 'fsArea_R_TF_ROI',
    'fsVol_L_TF_ROI', 'fsVol_R_TF_ROI', 'fsLGI_L_TF_ROI', 'fsLGI_R_TF_ROI',
    'fsCT_L_TGd_ROI', 'fsCT_R_TGd_ROI', 'fsArea_L_TGd_ROI', 'fsArea_R_TGd_ROI',
    'fsVol_L_TGd_ROI', 'fsVol_R_TGd_ROI', 'fsLGI_L_TGd_ROI', 'fsLGI_R_TGd_ROI',
    'fsCT_L_TGv_ROI', 'fsCT_R_TGv_ROI', 'fsArea_L_TGv_ROI', 'fsArea_R_TGv_ROI',
    'fsVol_L_TGv_ROI', 'fsVol_R_TGv_ROI', 'fsLGI_L_TGv_ROI', 'fsLGI_R_TGv_ROI',
    'fsCT_L_TPOJ1_ROI', 'fsCT_R_TPOJ1_ROI', 'fsArea_L_TPOJ1_ROI', 'fsArea_R_TPOJ1_ROI',
    'fsVol_L_TPOJ1_ROI', 'fsVol_R_TPOJ1_ROI', 'fsLGI_L_TPOJ1_ROI', 'fsLGI_R_TPOJ1_ROI',
    'fsCT_L_TPOJ2_ROI', 'fsCT_R_TPOJ2_ROI', 'fsArea_L_TPOJ2_ROI', 'fsArea_R_TPOJ2_ROI',
    'fsVol_L_TPOJ2_ROI', 'fsVol_R_TPOJ2_ROI', 'fsLGI_L_TPOJ2_ROI', 'fsLGI_R_TPOJ2_ROI',
    'fsCT_L_TPOJ3_ROI', 'fsCT_R_TPOJ3_ROI', 'fsArea_L_TPOJ3_ROI', 'fsArea_R_TPOJ3_ROI',
    'fsVol_L_TPOJ3_ROI', 'fsVol_R_TPOJ3_ROI', 'fsLGI_L_TPOJ3_ROI', 'fsLGI_R_TPOJ3_ROI',
    'fsCT_L_V1_ROI', 'fsCT_R_V1_ROI', 'fsArea_L_V1_ROI', 'fsArea_R_V1_ROI',
    'fsVol_L_V1_ROI', 'fsVol_R_V1_ROI', 'fsLGI_L_V1_ROI', 'fsLGI_R_V1_ROI',
    'fsCT_L_V2_ROI', 'fsCT_R_V2_ROI', 'fsArea_L_V2_ROI', 'fsArea_R_V2_ROI',
    'fsVol_L_V2_ROI', 'fsVol_R_V2_ROI', 'fsLGI_L_V2_ROI', 'fsLGI_R_V2_ROI',
    'fsCT_L_V3_ROI', 'fsCT_R_V3_ROI', 'fsArea_L_V3_ROI', 'fsArea_R_V3_ROI',
    'fsVol_L_V3_ROI', 'fsVol_R_V3_ROI', 'fsLGI_L_V3_ROI', 'fsLGI_R_V3_ROI',
    'fsCT_L_V3A_ROI', 'fsCT_R_V3A_ROI', 'fsArea_L_V3A_ROI', 'fsArea_R_V3A_ROI',
    'fsVol_L_V3A_ROI', 'fsVol_R_V3A_ROI', 'fsLGI_L_V3A_ROI', 'fsLGI_R_V3A_ROI',
    'fsCT_L_V3B_ROI', 'fsCT_R_V3B_ROI', 'fsArea_L_V3B_ROI', 'fsArea_R_V3B_ROI',
    'fsVol_L_V3B_ROI', 'fsVol_R_V3B_ROI', 'fsLGI_L_V3B_ROI', 'fsLGI_R_V3B_ROI',
    'fsCT_L_V3CD_ROI', 'fsCT_R_V3CD_ROI', 'fsArea_L_V3CD_ROI', 'fsArea_R_V3CD_ROI',
    'fsVol_L_V3CD_ROI', 'fsVol_R_V3CD_ROI', 'fsLGI_L_V3CD_ROI', 'fsLGI_R_V3CD_ROI',
    'fsCT_L_V4_ROI', 'fsCT_R_V4_ROI', 'fsArea_L_V4_ROI', 'fsArea_R_V4_ROI',
    'fsVol_L_V4_ROI', 'fsVol_R_V4_ROI', 'fsLGI_L_V4_ROI', 'fsLGI_R_V4_ROI',
    'fsCT_L_V4t_ROI', 'fsCT_R_V4t_ROI', 'fsArea_L_V4t_ROI', 'fsArea_R_V4t_ROI',
    'fsVol_L_V4t_ROI', 'fsVol_R_V4t_ROI', 'fsLGI_L_V4t_ROI', 'fsLGI_R_V4t_ROI',
    'fsCT_L_V6_ROI', 'fsCT_R_V6_ROI', 'fsArea_L_V6_ROI', 'fsArea_R_V6_ROI',
    'fsVol_L_V6_ROI', 'fsVol_R_V6_ROI', 'fsLGI_L_V6_ROI', 'fsLGI_R_V6_ROI',
    'fsCT_L_V6A_ROI', 'fsCT_R_V6A_ROI', 'fsArea_L_V6A_ROI', 'fsArea_R_V6A_ROI',
    'fsVol_L_V6A_ROI', 'fsVol_R_V6A_ROI', 'fsLGI_L_V6A_ROI', 'fsLGI_R_V6A_ROI',
    'fsCT_L_V7_ROI', 'fsCT_R_V7_ROI', 'fsArea_L_V7_ROI', 'fsArea_R_V7_ROI',
    'fsVol_L_V7_ROI', 'fsVol_R_V7_ROI', 'fsLGI_L_V7_ROI', 'fsLGI_R_V7_ROI',
    'fsCT_L_V8_ROI', 'fsCT_R_V8_ROI', 'fsArea_L_V8_ROI', 'fsArea_R_V8_ROI',
    'fsVol_L_V8_ROI', 'fsVol_R_V8_ROI', 'fsLGI_L_V8_ROI', 'fsLGI_R_V8_ROI',
    'fsCT_L_VIP_ROI', 'fsCT_R_VIP_ROI', 'fsArea_L_VIP_ROI', 'fsArea_R_VIP_ROI',
    'fsVol_L_VIP_ROI', 'fsVol_R_VIP_ROI', 'fsLGI_L_VIP_ROI', 'fsLGI_R_VIP_ROI',
    'fsCT_L_VMV1_ROI', 'fsCT_R_VMV1_ROI', 'fsArea_L_VMV1_ROI', 'fsArea_R_VMV1_ROI',
    'fsVol_L_VMV1_ROI', 'fsVol_R_VMV1_ROI', 'fsLGI_L_VMV1_ROI', 'fsLGI_R_VMV1_ROI',
    'fsCT_L_VMV2_ROI', 'fsCT_R_VMV2_ROI', 'fsArea_L_VMV2_ROI', 'fsArea_R_VMV2_ROI',
    'fsVol_L_VMV2_ROI', 'fsVol_R_VMV2_ROI', 'fsLGI_L_VMV2_ROI', 'fsLGI_R_VMV2_ROI',
    'fsCT_L_VMV3_ROI', 'fsCT_R_VMV3_ROI', 'fsArea_L_VMV3_ROI', 'fsArea_R_VMV3_ROI',
    'fsVol_L_VMV3_ROI', 'fsVol_R_VMV3_ROI', 'fsLGI_L_VMV3_ROI', 'fsLGI_R_VMV3_ROI',
    'fsCT_L_VVC_ROI', 'fsCT_R_VVC_ROI', 'fsArea_L_VVC_ROI', 'fsArea_R_VVC_ROI',
    'fsVol_L_VVC_ROI', 'fsVol_R_VVC_ROI', 'fsLGI_L_VVC_ROI', 'fsLGI_R_VVC_ROI',
    'fsCT_L_a10p_ROI', 'fsCT_R_a10p_ROI', 'fsArea_L_a10p_ROI', 'fsArea_R_a10p_ROI',
    'fsVol_L_a10p_ROI', 'fsVol_R_a10p_ROI', 'fsLGI_L_a10p_ROI', 'fsLGI_R_a10p_ROI',
    'fsCT_L_a24_ROI', 'fsCT_R_a24_ROI', 'fsArea_L_a24_ROI', 'fsArea_R_a24_ROI',
    'fsVol_L_a24_ROI', 'fsVol_R_a24_ROI', 'fsLGI_L_a24_ROI', 'fsLGI_R_a24_ROI',
    'fsCT_L_a24pr_ROI', 'fsCT_R_a24pr_ROI', 'fsArea_L_a24pr_ROI', 'fsArea_R_a24pr_ROI',
    'fsVol_L_a24pr_ROI', 'fsVol_R_a24pr_ROI', 'fsLGI_L_a24pr_ROI', 'fsLGI_R_a24pr_ROI',
    'fsCT_L_a32pr_ROI', 'fsCT_R_a32pr_ROI', 'fsArea_L_a32pr_ROI', 'fsArea_R_a32pr_ROI',
    'fsVol_L_a32pr_ROI', 'fsVol_R_a32pr_ROI', 'fsLGI_L_a32pr_ROI', 'fsLGI_R_a32pr_ROI',
    'fsCT_L_a47r_ROI', 'fsCT_R_a47r_ROI', 'fsArea_L_a47r_ROI', 'fsArea_R_a47r_ROI',
    'fsVol_L_a47r_ROI', 'fsVol_R_a47r_ROI', 'fsLGI_L_a47r_ROI', 'fsLGI_R_a47r_ROI',
    'fsCT_L_a9-46v_ROI', 'fsCT_R_a9-46v_ROI', 'fsArea_L_a9-46v_ROI', 'fsArea_R_a9-46v_ROI',
    'fsVol_L_a9-46v_ROI', 'fsVol_R_a9-46v_ROI', 'fsLGI_L_a9-46v_ROI', 'fsLGI_R_a9-46v_ROI',
    'fsCT_L_d23ab_ROI', 'fsCT_R_d23ab_ROI', 'fsArea_L_d23ab_ROI', 'fsArea_R_d23ab_ROI',
    'fsVol_L_d23ab_ROI', 'fsVol_R_d23ab_ROI', 'fsLGI_L_d23ab_ROI', 'fsLGI_R_d23ab_ROI',
    'fsCT_L_d32_ROI', 'fsCT_R_d32_ROI', 'fsArea_L_d32_ROI', 'fsArea_R_d32_ROI',
    'fsVol_L_d32_ROI', 'fsVol_R_d32_ROI', 'fsLGI_L_d32_ROI', 'fsLGI_R_d32_ROI',
    'fsCT_L_i6-8_ROI', 'fsCT_R_i6-8_ROI', 'fsArea_L_i6-8_ROI', 'fsArea_R_i6-8_ROI',
    'fsVol_L_i6-8_ROI', 'fsVol_R_i6-8_ROI', 'fsLGI_L_i6-8_ROI', 'fsLGI_R_i6-8_ROI',
    'fsCT_L_p10p_ROI', 'fsCT_R_p10p_ROI', 'fsArea_L_p10p_ROI', 'fsArea_R_p10p_ROI',
    'fsVol_L_p10p_ROI', 'fsVol_R_p10p_ROI', 'fsLGI_L_p10p_ROI', 'fsLGI_R_p10p_ROI',
    'fsCT_L_p24_ROI', 'fsCT_R_p24_ROI', 'fsArea_L_p24_ROI', 'fsArea_R_p24_ROI',
    'fsVol_L_p24_ROI', 'fsVol_R_p24_ROI', 'fsLGI_L_p24_ROI', 'fsLGI_R_p24_ROI',
    'fsCT_L_p24pr_ROI', 'fsCT_R_p24pr_ROI', 'fsArea_L_p24pr_ROI', 'fsArea_R_p24pr_ROI',
    'fsVol_L_p24pr_ROI', 'fsVol_R_p24pr_ROI', 'fsLGI_L_p24pr_ROI', 'fsLGI_R_p24pr_ROI',
    'fsCT_L_p32_ROI', 'fsCT_R_p32_ROI', 'fsArea_L_p32_ROI', 'fsArea_R_p32_ROI',
    'fsVol_L_p32_ROI', 'fsVol_R_p32_ROI', 'fsLGI_L_p32_ROI', 'fsLGI_R_p32_ROI',
    'fsCT_L_p32pr_ROI', 'fsCT_R_p32pr_ROI', 'fsArea_L_p32pr_ROI', 'fsArea_R_p32pr_ROI',
    'fsVol_L_p32pr_ROI', 'fsVol_R_p32pr_ROI', 'fsLGI_L_p32pr_ROI', 'fsLGI_R_p32pr_ROI',
    'fsCT_L_p47r_ROI', 'fsCT_R_p47r_ROI', 'fsArea_L_p47r_ROI', 'fsArea_R_p47r_ROI',
    'fsVol_L_p47r_ROI', 'fsVol_R_p47r_ROI', 'fsLGI_L_p47r_ROI', 'fsLGI_R_p47r_ROI',
    'fsCT_L_p9-46v_ROI', 'fsCT_R_p9-46v_ROI', 'fsArea_L_p9-46v_ROI', 'fsArea_R_p9-46v_ROI',
    'fsVol_L_p9-46v_ROI', 'fsVol_R_p9-46v_ROI', 'fsLGI_L_p9-46v_ROI', 'fsLGI_R_p9-46v_ROI',
    'fsCT_L_pOFC_ROI', 'fsCT_R_pOFC_ROI', 'fsArea_L_pOFC_ROI', 'fsArea_R_pOFC_ROI',
    'fsVol_L_pOFC_ROI', 'fsVol_R_pOFC_ROI', 'fsLGI_L_pOFC_ROI', 'fsLGI_R_pOFC_ROI',
    'fsCT_L_s32_ROI', 'fsCT_R_s32_ROI', 'fsArea_L_s32_ROI', 'fsArea_R_s32_ROI',
    'fsVol_L_s32_ROI', 'fsVol_R_s32_ROI', 'fsLGI_L_s32_ROI', 'fsLGI_R_s32_ROI',
    'fsCT_L_s6-8_ROI', 'fsCT_R_s6-8_ROI', 'fsArea_L_s6-8_ROI', 'fsArea_R_s6-8_ROI',
    'fsVol_L_s6-8_ROI', 'fsVol_R_s6-8_ROI', 'fsLGI_L_s6-8_ROI', 'fsLGI_R_s6-8_ROI',
    'fsCT_L_v23ab_ROI', 'fsCT_R_v23ab_ROI', 'fsArea_L_v23ab_ROI', 'fsArea_R_v23ab_ROI',
    'fsVol_L_v23ab_ROI', 'fsVol_R_v23ab_ROI', 'fsLGI_L_v23ab_ROI', 'fsLGI_R_v23ab_ROI',
]

_ABIDE2_TSV_URL = 'https://raw.githubusercontent.com/neurohackademy/nh2020-curriculum/e4eed3c4daa7f40b0ba931182a8c7e5e691dba6b/tu-machine-learning-yarkoni/data/abide2.tsv'
_ABIDE2_TSV_SHA256 = 'ad0db42f85c77cbb8b4e023df288d34b29f1a63870a0cf1955c5b3b0d357579e'


def load_abide_structural_table():
    """Fetch the pinned ABIDE-II structural table (network required) and
    return a DataFrame with the 1,440 structural predictors (cortical
    thickness, surface area, grey-matter volume, local gyrification index)
    plus "group" (1=autism, 2=control). Verifies the download's SHA-256
    against the pinned value before using it -- this is the same source
    book/config/abide_modeling.json already pins for Exercises 2-9, just
    with three more feature families included.
    """
    with urllib.request.urlopen(_ABIDE2_TSV_URL, timeout=180) as response:
        raw = response.read()
    digest = hashlib.sha256(raw).hexdigest()
    if digest != _ABIDE2_TSV_SHA256:
        raise RuntimeError(
            f"downloaded abide2.tsv sha256 {digest} does not match the pinned "
            f"{_ABIDE2_TSV_SHA256}. Refusing to use unexpected bytes."
        )
    table = pd.read_csv(io.BytesIO(raw), sep="\t")
    table = table.loc[:, [c for c in table.columns if not str(c).startswith("Unnamed")]]
    return table[_FEATURE_COLUMNS + ["group"]].reset_index(drop=True)


_QUESTION_CSS = """
<style>
.checked-question .widget-radio-box { width: 100% !important; }
.checked-question .widget-radio-box label {
    width: 100% !important;
    max-width: 100% !important;
    white-space: normal !important;
    height: auto !important;
    box-sizing: border-box;
    padding: 3px 0;
    line-height: 1.3;
}
</style>
"""


def _question_style_widget():
    return widgets.HTML(_QUESTION_CSS, layout=widgets.Layout(height="0px", margin="0px", padding="0px"))


def make_single_choice_question(prompt, options, correct_index, feedback_correct="Correct.", feedback_incorrect="Not quite -- try again."):
    """A compact single-choice checked question. Returns a widget to display()."""
    radio = widgets.RadioButtons(options=list(options), description="", layout=widgets.Layout(width="100%"))
    button = widgets.Button(description="Check answer", button_style="primary")
    feedback = widgets.Output()

    def _on_click(_button):
        with feedback:
            feedback.clear_output(wait=True)
            print(feedback_correct if radio.index == correct_index else feedback_incorrect)

    button.on_click(_on_click)
    box = widgets.VBox(
        [_question_style_widget(), widgets.HTML(f"<b>{prompt}</b>"), radio, button, feedback],
        layout=widgets.Layout(width="100%"),
    )
    box.add_class("checked-question")
    return box


def plot_learning_curves(train_losses, val_losses, best_epoch):
    fig, ax = plt.subplots(figsize=(6, 4))
    epochs = range(1, len(train_losses) + 1)
    ax.plot(epochs, train_losses, label="training loss", marker="o", markersize=3)
    ax.plot(epochs, val_losses, label="validation loss", marker="o", markersize=3)
    ax.axvline(best_epoch, color="0.4", linestyle="--", linewidth=1, label=f"best epoch ({best_epoch})")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("BCEWithLogitsLoss")
    ax.set_title("Training and validation loss by epoch")
    ax.legend()
    fig.tight_layout()
    return fig, ax


def confusion_counts(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    return tn, fp, fn, tp


def classification_report_row(name, y_true, probs):
    from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score

    pred = (np.asarray(probs) >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_counts(y_true, pred)
    sensitivity = tp / (tp + fn) if (tp + fn) else float("nan")
    specificity = tn / (tn + fp) if (tn + fp) else float("nan")
    try:
        auc = roc_auc_score(y_true, probs)
    except ValueError:
        auc = float("nan")
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "sensitivity": sensitivity,
        "specificity": specificity,
        "roc_auc": auc,
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
    }


_QUESTIONS = {
    "q-tensor-shape": {
        "shuffle_seed": 3,
        "prompt": "X_train_t has shape (602, 1440) and y_train_t has shape (602, 1). What does the first dimension of each tensor represent, and why must the two match?",
        "options": [
            "The number of participants in the training set; each row of X_train_t and the matching row of y_train_t describe the same participant, so a prediction can be matched to the right label.",
            "The number of structural features; it must match because PyTorch requires inputs and labels to have the same number of columns.",
            "The batch size used by the DataLoader; it is unrelated to how many participants are in the training set."
        ],
        "correct_index": 0,
        "feedback_correct": "Correct: the first dimension is the number of rows (participants). X and y must agree on it so every feature row lines up with its own label."
    },
    "q-logits-loss": {
        "shuffle_seed": 8,
        "prompt": "The network's final layer outputs one raw logit per participant, with no sigmoid applied before BCEWithLogitsLoss. Why?",
        "options": [
            "BCEWithLogitsLoss combines a sigmoid and binary cross-entropy internally, in a way that is more numerically stable than computing sigmoid and the loss as two separate steps.",
            "Logits are required because scikit-learn's LogisticRegression also outputs logits.",
            "Applying a sigmoid before this loss would make the gradients too large for Adam to handle."
        ],
        "correct_index": 0,
        "feedback_correct": "Correct: BCEWithLogitsLoss's internal log-sum-exp formulation avoids the numerical instability of computing sigmoid and log-loss separately."
    },
    "q-gradient-update": {
        "shuffle_seed": 13,
        "prompt": "Suppose optimizer.zero_grad() were accidentally removed from train_one_epoch, leaving loss.backward() and optimizer.step() in place. What would happen across batches?",
        "options": [
            "Gradients from each new batch would accumulate on top of gradients from every previous batch, so each update would reflect more than just the current batch.",
            "Nothing would change; zero_grad() only matters for the very first batch of training.",
            "Training would stop immediately with a shape-mismatch error."
        ],
        "correct_index": 0,
        "feedback_correct": "Correct: PyTorch accumulates gradients by default. Without zero_grad(), .grad keeps growing across batches instead of reflecting only the current one."
    },
    "q-train-only-scaling": {
        "shuffle_seed": 21,
        "prompt": "SimpleImputer and StandardScaler are both fit on X_train only, then used only to transform X_val and X_test. Why not fit them on all 1,004 participants before splitting?",
        "options": [
            "Fitting on the full dataset would let information about validation/test rows (their means and scales) influence what the model trains on, overstating how well it will generalize to genuinely new participants.",
            "Fitting on the full dataset is statistically fine; the course only avoids it to save computation time.",
            "StandardScaler raises an error if it is fit on more than 1,000 rows."
        ],
        "correct_index": 0,
        "feedback_correct": "Correct: fitting preprocessing on rows the model will later be evaluated on is a form of data leakage, even though no label is involved."
    },
    "q-overfitting-earlystop": {
        "shuffle_seed": 6,
        "prompt": "During training, train_loss kept decreasing every epoch, but val_loss decreased for a few epochs and then started rising. What does the early-stopping rule in this notebook do in response?",
        "options": [
            "It keeps the model weights from whichever epoch had the lowest validation loss, rather than the weights from the final epoch -- the rising val_loss after that point indicates the network is starting to overfit the training rows.",
            "It increases the batch size automatically until val_loss also decreases.",
            "It is expected and nothing is done about it; the final epoch's weights are always used regardless of val_loss."
        ],
        "correct_index": 0,
        "feedback_correct": "Correct: a falling train_loss with a rising val_loss is the classic overfitting signature. Restoring the best-validation-epoch weights avoids reporting an overfit final state."
    },
    "q-test-set-choices": {
        "shuffle_seed": 17,
        "prompt": "This notebook chose the number of epochs (via early stopping) and the hidden-layer sizes using only X_val/y_val, then evaluated on X_test exactly once. Why not simply try several configurations and keep whichever one scores best on X_test?",
        "options": [
            "Using the test set to choose among configurations turns it into another validation set; the reported test score would then overestimate how the chosen model performs on truly new participants.",
            "It would work just as well; the course avoids it only because scikit-learn's train_test_split does not support reusing a split twice.",
            "The test set is too small for this; a larger one could be used to choose hyperparameters safely."
        ],
        "correct_index": 0,
        "feedback_correct": "Correct: any choice made by looking at test performance -- even just picking among a few configurations -- contaminates the test set's role as an honest, one-time check."
    }
}


def show_question(question_id):
    q = _QUESTIONS[question_id]
    order = list(range(len(q["options"])))
    random.Random(q["shuffle_seed"]).shuffle(order)
    shuffled_options = [q["options"][i] for i in order]
    shuffled_correct = order.index(q["correct_index"])
    display(make_single_choice_question(
        q["prompt"], shuffled_options, shuffled_correct, q.get("feedback_correct", "Correct."),
    ))


## 1. Setup and the Task


**The task.** Classify each participant in the ABIDE-II dataset as autism
or control, using several structural MRI measures derived from each
participant's cortical surface. This is the same kind of classification
task as Exercise 3, but with more feature families available to the model.


In [ ]:
try:
    import torch
except ImportError:
    %pip install torch
    import torch

import torch.nn as nn

print("PyTorch version:", torch.__version__)


### Loading the data

The table below comes from the same pinned ABIDE-II source
(NeuroHackademy/Yarkoni tutorial, CC-BY-4.0) that Exercises 2-9 use. Those
earlier notebooks used only one structural measure, **cortical thickness**
(360 columns: 180 brain regions x 2 hemispheres). Here we add three more
structural measures derived from the same cortical surface reconstruction:
**cortical surface area**, **grey-matter volume**, and **local
gyrification index** (a measure of how folded the cortical surface is).
Together, that is 4 measures x 180 regions x 2 hemispheres = 1,440
structural predictors, for the same 1,004 participants.


In [ ]:
try:
    df = load_abide_structural_table()
except NameError as exc:
    raise RuntimeError(
        "load_abide_structural_table is not defined yet. Run this "
        "notebook's first code cell (collapsed, at the very top, under "
        "'Run the cell below first') before this one, then run this cell again."
    ) from exc

X_all = df[_FEATURE_COLUMNS].to_numpy(dtype="float64")
y_all = (df["group"].to_numpy() == 1).astype("float64")  # autism=1, control=0

n_autism = int(y_all.sum())
n_control = int((y_all == 0).sum())
print(f"participants: {df.shape[0]}")
print(f"structural features: {X_all.shape[1]} (4 families x 180 regions x 2 hemispheres)")
print(f"class counts: autism={n_autism} ({n_autism/len(y_all):.1%})  control={n_control} ({n_control/len(y_all):.1%})")
print(f"missing values in X_all: {int(pd.isna(X_all).sum())}")


This table has **zero missing values** in these structural columns. Real
neuroimaging tables are not always this complete, so Section 2 still fits
a missing-value step as part of the standard recipe -- it simply will have
nothing to do here.

> **Note on the JupyterLite limitation (Gate A).** This course's browser
> notebook platform (JupyterLite) runs Python inside the browser via
> Pyodide. The exact Pyodide build this course currently pins
> (jupyterlite-pyodide-kernel 0.8.5, Pyodide v314.0.5) does not include
> PyTorch in its package set, and PyPI has no PyTorch wheel for that
> WebAssembly platform either -- so `pip install torch` cannot succeed
> there. This was verified directly against that build, not assumed. The
> course author will decide separately whether and how to offer this
> exercise on the live site; for now, use Colab or local Jupyter.


## 2. Hold-Out Design and Tensors


We split the 1,004 participants into three groups **before** touching any
feature scaling: a training set, a validation set (used to make decisions
about the network and its training), and a test set that is evaluated
exactly once, at the very end, in Section 5. All three splits are
stratified by diagnosis so each group has a similar autism/control
balance.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_rest, y_train, y_rest = train_test_split(
    X_all, y_all, test_size=0.4, random_state=42, stratify=y_all
)
X_val, X_test, y_val, y_test = train_test_split(
    X_rest, y_rest, test_size=0.5, random_state=42, stratify=y_rest
)

print(f"X_train: {X_train.shape}   X_val: {X_val.shape}   X_test: {X_test.shape}")
for name, y in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"  {name}: autism={int(y.sum())}  control={int((y == 0).sum())}")


### Missing values and scaling -- fit on training rows only

`SimpleImputer` and `StandardScaler` are both **fit on `X_train` only**,
then used to *transform* `X_val` and `X_test`. Fitting either step on
validation or test rows would let information about those rows (their
means, their scales) leak into numbers the model is trained on --
overstating how well the model will generalize to genuinely new
participants.


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

imputer = SimpleImputer(strategy="mean").fit(X_train)
X_train_i = imputer.transform(X_train)
X_val_i = imputer.transform(X_val)
X_test_i = imputer.transform(X_test)

scaler = StandardScaler().fit(X_train_i)
X_train_s = scaler.transform(X_train_i).astype("float32")
X_val_s = scaler.transform(X_val_i).astype("float32")
X_test_s = scaler.transform(X_test_i).astype("float32")

print("scaled training features: mean", round(float(X_train_s.mean()), 4), "std", round(float(X_train_s.std()), 4))


### Tensors and batches

Now turn the scaled NumPy arrays into PyTorch tensors, and group the
training rows into batches with a `DataLoader`. A `TensorDataset` pairs
each feature row with its label; a `DataLoader` then serves them in
shuffled batches of a fixed size (`BATCH_SIZE`), which is what the
training loop in Section 4 will iterate over.

**Your task:** convert `X_train_s`/`X_val_s`/`X_test_s` and
`y_train`/`y_val`/`y_test` into `float32` tensors (labels as column
vectors, shape `(n, 1)`, since `BCEWithLogitsLoss` expects predictions and
targets to have matching shapes), then build a `TensorDataset` and
`DataLoader` for the training and validation sets.


In [ ]:
BATCH_SIZE = 64

X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)

X_val_t = torch.tensor(X_val_s, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32).reshape(-1, 1)

X_test_t = torch.tensor(X_test_s, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

train_dataset = torch.utils.data.TensorDataset(X_train_t, y_train_t)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

val_dataset = torch.utils.data.TensorDataset(X_val_t, y_val_t)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [ ]:
# Run this to check your tensors and batches.
_required = ["X_train_t", "y_train_t", "X_val_t", "y_val_t", "X_test_t", "y_test_t", "train_loader", "val_loader"]
_missing = [name for name in _required if name not in globals() or globals()[name] is Ellipsis]
if _missing:
    print(f"Not finished yet -- these are still missing or unfilled: {', '.join(_missing)}.")
else:
    problems = []
    if X_train_t.dtype != torch.float32 or y_train_t.dtype != torch.float32:
        problems.append("X_train_t and y_train_t must both have dtype torch.float32.")
    if y_train_t.ndim != 2 or y_train_t.shape[1] != 1:
        problems.append(f"y_train_t should have shape (n, 1); got {tuple(y_train_t.shape)}.")
    if X_train_t.shape[0] != y_train_t.shape[0]:
        problems.append("X_train_t and y_train_t must have the same number of rows.")
    if not torch.isfinite(X_train_t).all():
        problems.append("X_train_t contains non-finite values (NaN or inf).")
    if problems:
        for p in problems:
            print("Not quite:", p)
    else:
        xb, yb = next(iter(train_loader))
        print(f"One training batch: xb.shape={tuple(xb.shape)}  yb.shape={tuple(yb.shape)}  yb unique values={sorted(set(yb.flatten().tolist()))}")
        print("Looks good: tensors are float32, shapes line up, and values are finite.")


## 3. Build a Fully Connected Network


Now define the network: an `nn.Module` with two modest hidden layers
(128 units, then 32 units -- chosen, along with the learning
rate below, by comparing a handful of configurations on the *validation*
set only, never the test set) and a nonlinear activation (`ReLU`) between
layers. The final layer outputs **one raw logit per participant** -- not a
probability. We pass that logit directly to `nn.BCEWithLogitsLoss`, which
combines a sigmoid and binary cross-entropy in one numerically stable step;
applying `sigmoid` ourselves before this loss would double up that step and
is a common mistake to avoid.

**Your task:** fill in the three linear layers (sizes are given in the
`__init__` signature) and the `forward` method that chains them together
with the activations.


In [ ]:
class AutismClassifierNet(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden1)
        self.act1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.act2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden2, 1)

    def forward(self, x):
        x = self.act1(self.fc1(x))
        x = self.act2(self.fc2(x))
        return self.fc3(x)


In [ ]:
# Run this to check your model definition.
if "AutismClassifierNet" not in globals():
    print("Not finished yet -- AutismClassifierNet is not defined.")
else:
    try:
        _probe = AutismClassifierNet(X_train_t.shape[1], 128, 32)
        _out = _probe(X_train_t[:5])
    except Exception as exc:
        print(f"Not quite -- building or running the model raised an error: {exc}")
    else:
        if tuple(_out.shape) != (5, 1):
            print(f"Not quite: forward(X_train_t[:5]) should have shape (5, 1); got {tuple(_out.shape)}.")
        else:
            print(f"Looks good: forward() on 5 rows returns shape {tuple(_out.shape)} (one logit per row).")


## 4. Train and Validate


A training epoch repeats, for every batch: a **forward pass** (compute
predictions), compute the **loss**, reset gradients from the previous
batch (`optimizer.zero_grad()`), **backpropagate** (`loss.backward()`,
which fills in `.grad` on every parameter), then take an **optimizer
step** (`optimizer.step()`, which updates the parameters using those
gradients). `model.train()` and `model.eval()` switch layers like dropout
or batch normalization between their training and inference behavior --
this model has neither, but setting the mode explicitly is good practice
and is required once you add layers that do depend on it.

**Your task:** complete the four missing lines below: the forward pass and
loss inside `train_one_epoch`, and the forward pass and loss inside
`evaluate`.


In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn):
    model.train()
    running_loss = 0.0
    n_rows = 0
    for xb, yb in loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
        n_rows += xb.size(0)
    return running_loss / n_rows


def evaluate(model, loader, loss_fn):
    model.eval()
    running_loss = 0.0
    n_rows = 0
    all_probs, all_targets = [], []
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb)
            loss = loss_fn(logits, yb)
            running_loss += loss.item() * xb.size(0)
            n_rows += xb.size(0)
            all_probs.append(torch.sigmoid(logits))
            all_targets.append(yb)
    probs = torch.cat(all_probs).numpy().ravel()
    targets = torch.cat(all_targets).numpy().ravel()
    return running_loss / n_rows, probs, targets


In [ ]:
# Run this to check your training/evaluation functions on one epoch.
if "train_one_epoch" not in globals() or "evaluate" not in globals():
    print("Not finished yet -- train_one_epoch and/or evaluate are not defined.")
else:
    try:
        _probe_model = AutismClassifierNet(X_train_t.shape[1], HIDDEN_1, HIDDEN_2) if "HIDDEN_1" in globals() else AutismClassifierNet(X_train_t.shape[1], 128, 32)
        _probe_loss = nn.BCEWithLogitsLoss()
        _probe_opt = torch.optim.Adam(_probe_model.parameters(), lr=1e-3)
        _train_loss = train_one_epoch(_probe_model, train_loader, _probe_opt, _probe_loss)
        _val_loss, _val_probs, _val_targets = evaluate(_probe_model, val_loader, _probe_loss)
    except Exception as exc:
        print(f"Not quite -- running one epoch raised an error: {exc}")
    else:
        problems = []
        if not (0.0 < _train_loss < 5.0):
            problems.append(f"train_one_epoch returned an unusual loss value ({_train_loss:.4f}); expected a small positive number.")
        if not (_val_probs.min() >= 0.0 and _val_probs.max() <= 1.0):
            problems.append("evaluate's returned probabilities should all be between 0 and 1 (did you apply sigmoid?).")
        if len(_val_probs) != len(y_val):
            problems.append(f"evaluate returned {len(_val_probs)} probabilities; expected one per validation row ({len(y_val)}).")
        if problems:
            for p in problems:
                print("Not quite:", p)
        else:
            print(f"Looks good: one training epoch ran (train_loss={_train_loss:.4f}), and evaluate() returned {len(_val_probs)} valid probabilities.")


### Training with early stopping

The cell below trains the real model for up to `MAX_EPOCHS` epochs, using
your `train_one_epoch` and `evaluate` functions, with a fixed random seed
for reproducibility. After every epoch, it checks the validation loss: if
it has not improved for `PATIENCE` epochs in a row, training stops early,
and the model's weights are rolled back to whichever epoch had the best
(lowest) validation loss -- not necessarily the final epoch. This keeps an
overfit final state from being reported as "the" trained model.

This cell trains the network exactly once per normal run of this notebook
-- not on every edit of the cells above it.


In [ ]:
import random

_required = ["AutismClassifierNet", "train_one_epoch", "evaluate", "train_loader", "val_loader", "X_train_t"]
_missing = [name for name in _required if name not in globals() or globals()[name] is Ellipsis]
if _missing:
    raise RuntimeError(
        "Not ready yet -- go back and complete the earlier blank(s) for: "
        + ", ".join(_missing)
        + " (Sections 2-4) before running this cell."
    )

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

HIDDEN_1 = 128
HIDDEN_2 = 32
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0001
MAX_EPOCHS = 50
PATIENCE = 8
LOGREG_C = 0.003  # used later, in Section 5's baseline comparison

model = AutismClassifierNet(X_train_t.shape[1], HIDDEN_1, HIDDEN_2)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

train_losses, val_losses = [], []
best_val_loss = float("inf")
best_state = None
best_epoch = -1
epochs_no_improve = 0

for epoch in range(MAX_EPOCHS):
    train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn)
    val_loss, _, _ = evaluate(model, val_loader, loss_fn)
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        best_epoch = epoch
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= PATIENCE:
        print(f"Early stopping after epoch {epoch + 1} (best epoch: {best_epoch + 1}, best val_loss: {best_val_loss:.4f})")
        break
else:
    print(f"Reached MAX_EPOCHS={MAX_EPOCHS} without triggering early stopping (best epoch: {best_epoch + 1})")

model.load_state_dict(best_state)
model.eval()
print(f"Restored weights from epoch {best_epoch + 1} (lowest validation loss: {best_val_loss:.4f}).")


In [ ]:
fig, ax = plot_learning_curves(train_losses, val_losses, best_epoch + 1)
plt.show()


In [ ]:
# A quick sanity check on the training run.
problems = []
if len(train_losses) < 2:
    problems.append("training ran for fewer than 2 epochs; something stopped it too early.")
if min(val_losses) != best_val_loss:
    problems.append("best_val_loss does not match the minimum of val_losses; check the early-stopping bookkeeping.")
if train_losses[-1] >= train_losses[0]:
    problems.append("training loss did not decrease at all from the first to the last epoch run -- check train_one_epoch.")
if problems:
    for p in problems:
        print("Check this:", p)
else:
    print(f"Looks good: trained for {len(train_losses)} epoch(s), training loss fell from {train_losses[0]:.4f} to {train_losses[-1]:.4f}, "
          f"and the restored weights are from the best validation epoch ({best_epoch + 1}).")


## 5. Final Evaluation and Comparison


Now -- for the first and only time -- evaluate the trained network on
`X_test`/`y_test`. This happens exactly once, after every choice about the
network (architecture, learning rate, number of epochs) was already made
using only the training and validation sets.

We compare three things on the same test participants, with the same 1,440
features: a **dummy baseline** that always predicts the majority class seen
in training, a **regularized logistic regression** fit on the same
train-only-scaled features (its regularization strength was likewise
chosen by checking validation performance across a small grid, never the
test set), and the **trained network**.


In [ ]:
with torch.no_grad():
    test_logits = model(X_test_t)
    test_probs_nn = torch.sigmoid(test_logits).numpy().ravel()

from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(C=LOGREG_C, max_iter=5000, random_state=42).fit(X_train_s, y_train)
test_probs_logreg = logreg.predict_proba(X_test_s)[:, 1]

majority_class = 0.0 if (y_train == 0).sum() >= (y_train == 1).sum() else 1.0
test_probs_dummy = np.full_like(y_test, majority_class)

rows = [
    classification_report_row("dummy (majority class)", y_test, test_probs_dummy),
    classification_report_row(f"logistic regression (C={LOGREG_C})", y_test, test_probs_logreg),
    classification_report_row("neural network", y_test, test_probs_nn),
]
comparison_table = pd.DataFrame(rows).set_index("model")
comparison_table


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
metrics_to_plot = ["accuracy", "balanced_accuracy", "roc_auc"]
x = np.arange(len(comparison_table))
width = 0.25
for i, metric in enumerate(metrics_to_plot):
    ax.bar(x + i * width, comparison_table[metric], width, label=metric)
ax.axhline(0.5, color="0.5", linestyle="--", linewidth=1, label="chance")
ax.set_xticks(x + width)
ax.set_xticklabels(comparison_table.index, rotation=15, ha="right")
ax.set_ylim(0, 1)
ax.set_title("Test-set comparison (evaluated once)")
ax.legend()
fig.tight_layout()
plt.show()


### Reading this comparison honestly

Report whatever `comparison_table` actually shows -- do not expect the
network to beat the baseline, and do not expect either model to clear 60%
accuracy. With roughly 600 training participants and 1,440 features, both
models are working with a hard problem: more features than this course's
earlier cortical-thickness-only models had, but not more participants, and
autism spectrum disorder's structural MRI signature is subtle and
heterogeneous across sites and individuals. For context only -- **not** as
a directly controlled comparison, since the feature set, split, and model
all differ -- Exercise 3's cortical-thickness-only logistic regression on
this same `group` target reached 54.6% test
accuracy (AUC 0.569).

> **Note on exact numbers.** Every number in this section is computed live
> by the cells above and may differ slightly from one PyTorch version or
> machine to another, even with the fixed random seed used here -- CPU
> floating-point operations are not guaranteed bit-identical across
> hardware/library versions. Expect similar magnitude and direction, not an
> exact digit match to any number printed in this cell's own text.


## 6. Interpretation and Limits


**Check your understanding.**


In [ ]:
show_question("q-tensor-shape")


In [ ]:
show_question("q-logits-loss")


In [ ]:
show_question("q-gradient-update")


In [ ]:
show_question("q-train-only-scaling")


In [ ]:
show_question("q-overfitting-earlystop")


In [ ]:
show_question("q-test-set-choices")


### What this does and does not show

A fully connected network can model nonlinear interactions between
features that a linear model like logistic regression cannot represent
directly, and in principle it can make better use of more data and more
features as both grow. But this cohort is small (1,004 participants spread
across 17 sites) and clinically and biologically heterogeneous, and adding
features without adding participants gives a flexible model more ways to
fit noise in the training set rather than real signal.

Structural MRI measures gathered across many sites and scanners are also
known to carry site/scanner effects that compete with the signal of
interest. Saponaro et al. (2022, *NeuroImage: Clinical*) harmonized 221
structural morphometric features (cortical thickness, grey-matter volume,
white-matter volume, and subcortical volumes from FreeSurfer) across 1,638
participants from 37 sites spanning ABIDE I and II, and found that a
classifier's cross-validated AUC rose from 0.58 before harmonization to
0.67 after it -- the same underlying case-control signal, substantially
easier to recover once multi-site batch effects were removed. This
notebook's data is not harmonized across its 17 sites, which is one
plausible, concrete reason neither model here reaches a high score; it is
a structural-MRI finding, distinct from the motion/data-quality literature
specific to **functional** MRI.

None of this supports a diagnostic or causal claim about autism from these
features, and a middling score here does not mean this general approach
has hit some definitive ceiling -- better harmonization, more
participants, or different features could change these numbers. Report
what you actually measured, not what you expected to measure.
